### Filtering with a function

In [3]:
import apache_beam as beam

garden_plants = [
    {'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'},
    {'icon': '🥕', 'name': 'Carrot', 'duration': 'biennial'},
    {'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'},
    {'icon': '🍅', 'name': 'Tomato', 'duration': 'annual'},
    {'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'},
]

def is_perennial(plant):
    return plant['duration'] == 'perennial'

with beam.Pipeline() as pipe:
    perennials = (
        pipe
        | 'Gardening plants' >> beam.Create(garden_plants)
        | 'Filter perennials' >> beam.Filter(is_perennial)
        | beam.Map(print)
    )

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}


### Filtering with a Lambda function

In [4]:
with beam.Pipeline() as pipe:
    perennials = (
        pipe
        | beam.Create(garden_plants)
        | 'Filter perennials' >> beam.Filter(lambda x: x['duration'] == 'perennial')
        | 'Prints' >> beam.Map(print)
    )

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}


### Filtering with multiple arguments

In [5]:
import apache_beam as beam

def has_duration(plant, duration):
    return plant['duration'] == duration

with beam.Pipeline() as pipe:
    perennials = (
        pipe
        | 'Gardening plants' >> beam.Create(garden_plants)
        | 'Filtering perennials' >> beam.Filter(has_duration, 'perennial')
        | beam.Map(print)
    )

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}


### Filtering with side inputs as singletons

If the `PCollection` has a single value, such as the average from another computation,
passing the `PCollection` as a *singleton* accesses that value.

In [9]:
import apache_beam as beam

with beam.Pipeline() as pipeline:
    pen = (pipeline | 'pen' >> beam.Create(["perennial"]))

    perennials = (
        pipeline
        | beam.Create(garden_plants)
        | 'Filtering perennials' >> beam.Filter(
            lambda x, duration:
            x['duration'] == duration,
            duration=beam.pvalue.AsSingleton(pen)
        )
        | beam.Map(print)
    )

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}


### Filtering with side inputs as iterators

In [10]:
import apache_beam as beam

with beam.Pipeline() as pipe:
    valid_durations = (
        pipe
        | "valid durations" >> beam.Create(['annual', 'biennial', 'perennial'])
    )

    valid_plant = (
        pipe 
        | 'Gardening plants' >> beam.Create(garden_plants)
        | "Filtering valid plants" >> beam.Filter(
            lambda x, valid:
            x['duration'] in valid,
            valid = beam.pvalue.AsIter(valid_durations)
        )
        | beam.Map(print)
    )

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🥕', 'name': 'Carrot', 'duration': 'biennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🍅', 'name': 'Tomato', 'duration': 'annual'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}


### Filtering with side inputs as dictionaries

In [11]:
import apache_beam as beam

with beam.Pipeline() as pipe:
    keep_duration = (
        pipe
        | 'Duration filters' >> beam.Create([
            ('annual', False),
            ('biennial', False),
            ('perennial', True)
        ])
    )

    perennials = (
        pipe 
        | "Gardeing plants" >> beam.Create(garden_plants)
        | 'Filter plants by duration' >> beam.Filter(
            lambda x, keep:
            keep[x['duration']],
            keep=beam.pvalue.AsDict(keep_duration)
        )
        | beam.Map(print)
    )

{'icon': '🍓', 'name': 'Strawberry', 'duration': 'perennial'}
{'icon': '🍆', 'name': 'Eggplant', 'duration': 'perennial'}
{'icon': '🥔', 'name': 'Potato', 'duration': 'perennial'}
